In [1]:
!pip install -q pinecone langchain langchain-pinecone langchain-core langchain-community langchain_google_genai pymupdf tiktoken python-dotenv langchain_huggingface sentence-transformers transformers pdfplumber

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader,PDFPlumberLoader
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_18340/1045642509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader,PDFPlumberLoader


# Document Ingestion

In [3]:
# loader = DirectoryLoader(
#     path="/content/Handbook",
#     glob = "*.pdf",
#     loader_cls = PyMuPDFLoader,
#     silent_errors=True
# )

loader = PDFPlumberLoader('/content/kiit_rules_updated_merged.pdf')
docs = loader.load()
print(len(docs))

287


In [4]:
print(docs[0].page_content)

SCHOOLOF COMPUTERENGINEERING,
KALINGA INSTITUTE OFINDUSTRIAL TECHNOLOGY
DEEMEDTOBE UNIVERSITY
Faculty Chamber allocation
Sl.
Name Email ID RoomNo.
No.
1 Mr. ARanjith a.ranjithfcs@kiit.ac.in B321E
2 Dr. Abhaya Kumar Sahoo abhaya.sahoofcs@kiit.ac.in C201A
3 Mr. AbhijitDeuri abhijit.deurifcs@kiit.ac.in A017D
4 Mr. Abhishek Raj abhishek.rajfcs@kiit.ac.in A217B
5 Prof. (Dr.) Abhishek Ray arayfcs@kiit.ac.in B005A
6 Mr. Abinas Panda abinas.pandafcs@kiit.ac.in B321C
7 Dr. Abir Sen abir.senfcs@kiit.ac.in B011D
8 Dr. Adyasha Dash adyasha.dashfcs@kiit.ac.in B121A
9 Mr. Ajay Anand ajay.anandfcs@kiit.ac.in A217G
C-203E
10 Dr. Ajay KumarJena ajay.jenafcs@kiit.ac.in
11 Dr. Ajaya Kumar Parida ajaya.paridafcs@kiit.ac.in A217E
12 Dr. Ajit Kumar Pasayat ajit.pasayatfcs@kiit.ac.in C301G
13 Dr. Aleena Swetapadma aleena.swetapadmafcs@kiit.ac.in B211A
Prof. (Dr.) Alok Kumar
C-101K
14 Jagadev alok.jagadevfcs@kiit.ac.in
15 Dr. AmbikaPrasad Mishra ambikaprasad.mishrafcs@kiit.ac.in A017A
16 Mr. AmitKumar amit.ku

In [5]:
docs[1].page_content

'Prof. (Dr.) Bhabani Shankar\nC201C\n33 Prasad Mishra bsmishrafcs@kiit.ac.in\n34 Dr. Bhaswati Sahoo bhaswati.sahoofcs@kiit.ac.in A305C\n35 Dr. Bindu Agarwalla bindu.agarwalfcs@kiit.ac.in A205E\n36 Mr. BirajaPrasad Nayak biraja.nayakfcs@kiit.ac.in A317G\nC001\n37 Prof.(Dr.)Biswajit Sahoo bsahoofcs@kiit.ac.in\n38 Dr. Biswajeet Sethi biswajeet.sethifcs@kiit.ac.in B111G\n39 Mrs. Chandani Kumari chandani.kumarifcs@kiit.ac.in A305B\n40 Dr. Chittaranjan Pradhan chittaranjanfcs@kiit.ac.in B221A\n41 Dr. Dayal Kumar Behera dayal.beherafcs@kiit.ac.in C301B\nB321F\n42 Dr. Debachudamani Prusti debachudamani.prustifcs@kiit.ac.in\n43 Dr. Debadatta Naik debadatta.naikfcs@kiit.ac.in B105H\n44 Dr. Debanjan Pathak debanjan.pathakfcs@kiit.ac.in A317B\n45 Mr. Debashis Hati dhatifcs@kiit.ac.in B205B\n46 Mr. Deependra Singh deependra.singhfcs@kiit.ac.in A017G\n47 Dr. Dipti Dash dipti.dashfcs@kiit.ac.in B021C\n48 Mr. Gananath Bhuyan gananatha.bhuyanfcs@kiit.ac.in A217F\nProf.(Dr.)Ganga Bishnu\nC301K\n49 Mund 

In [6]:
print(docs[10].metadata)

{'source': '/content/kiit_rules_updated_merged.pdf', 'file_path': '/content/kiit_rules_updated_merged.pdf', 'page': 10, 'total_pages': 287, 'Producer': 'iLovePDF', 'ModDate': 'D:20260804185939Z'}


Preprocessing the text for document

In [7]:
import re
def clean_text(text):

    text = re.sub(r'KIIT Publication Cell.*', '', text)

    text = re.sub(r'SOT Student Handbook.*', '', text)

    text = re.sub(r'Page\s+\d+', '', text)

    # text = re.sub(r'\b\d{10}\b','',text)

    text = re.sub(r'\n+', '\n', text)

    return text.strip()

In [8]:
for doc in docs:
    doc.page_content = clean_text(doc.page_content)

In [9]:
docs[1].page_content

'Prof. (Dr.) Bhabani Shankar\nC201C\n33 Prasad Mishra bsmishrafcs@kiit.ac.in\n34 Dr. Bhaswati Sahoo bhaswati.sahoofcs@kiit.ac.in A305C\n35 Dr. Bindu Agarwalla bindu.agarwalfcs@kiit.ac.in A205E\n36 Mr. BirajaPrasad Nayak biraja.nayakfcs@kiit.ac.in A317G\nC001\n37 Prof.(Dr.)Biswajit Sahoo bsahoofcs@kiit.ac.in\n38 Dr. Biswajeet Sethi biswajeet.sethifcs@kiit.ac.in B111G\n39 Mrs. Chandani Kumari chandani.kumarifcs@kiit.ac.in A305B\n40 Dr. Chittaranjan Pradhan chittaranjanfcs@kiit.ac.in B221A\n41 Dr. Dayal Kumar Behera dayal.beherafcs@kiit.ac.in C301B\nB321F\n42 Dr. Debachudamani Prusti debachudamani.prustifcs@kiit.ac.in\n43 Dr. Debadatta Naik debadatta.naikfcs@kiit.ac.in B105H\n44 Dr. Debanjan Pathak debanjan.pathakfcs@kiit.ac.in A317B\n45 Mr. Debashis Hati dhatifcs@kiit.ac.in B205B\n46 Mr. Deependra Singh deependra.singhfcs@kiit.ac.in A017G\n47 Dr. Dipti Dash dipti.dashfcs@kiit.ac.in B021C\n48 Mr. Gananath Bhuyan gananatha.bhuyanfcs@kiit.ac.in A217F\nProf.(Dr.)Ganga Bishnu\nC301K\n49 Mund 

# Chunking

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1300,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "•",
        ":"
    ]
)

ndocs = splitter.split_documents(docs)

print(ndocs[82].page_content)
print(len(ndocs))


 The fourth-year subjects carry only 30 credits. Three
more subjects can be easily accommodated in the fourth
year to earn a Minor.
 To earn a Major in a discipline other than the one in which a student is
registered, the student must secure at least 7.50 CGPA after the third
year, successfully complete at least 20-Credit coursework in that
discipline in the fourth year, in addition to fulfilling all the usual
fourth-year coursework requirements.
 To get two B. Tech (Hons.) or B. Tech. (Research) degrees, a student
fulfil all the coursework requirements of the original discipline in which
the student is registered and spend study another year or more to
complete at least 40 additional Credits of coursework in that discipline
in the fifth year.
5.3.3 The First-Year Curriculum (The Certificate Course)
The first-year curriculum provides the foundation for all the higher-level
courses in the next three and four years of engineering studies. Science
subjects, providing the required found

# Embedding the chunks

In [18]:
!pip install -q sentence-transformers

In [11]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

# Checking the embedding model for easy deploy

In [ ]:
from pinecone import Pinecone,ServerlessSpec
pc = Pinecone(api_key='')
index_name = "bandhu-db"

if pc.has_index(index_name):
    pc.delete_index(index_name)

pc.create_index(
    name = index_name,
    dimension = 1024,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "bandhu-db",
    "metric": "cosine",
    "host": "bandhu-db-7uweph7.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1024,
    "deletion_protection": "disabled",
    "tags": null
}

In [ ]:
import os
os.environ["PINECONE_API_KEY"] = ""

In [15]:
index_name = "bandhu-db"
vector_store = PineconeVectorStore(
    index_name = index_name,
    embedding= embedding_model
)

In [16]:
vector_store.add_documents(ndocs)

['1e682a34-5ec5-4930-a28e-9a88608a1c66',
 '17a8c529-3495-480d-930c-6968b4a13163',
 'b0747057-8358-486a-9788-960059cd5449',
 '909d24a6-f66f-408e-9ecf-63fd909c5e25',
 '293bf052-bf93-4655-a732-c61618727b17',
 'c0f648a1-d354-4925-809a-844e81028de0',
 '1f092e7a-cbe2-48fc-9449-159c8baa75c4',
 '13675425-79cf-4532-8cea-89d8180e2939',
 '0713c932-298f-4c69-963e-9b5b5572cff6',
 '513d6f40-1b39-41cb-8e0e-6fb6ae8e1148',
 'eed12f15-8dd9-48d9-a933-9e009c3a6762',
 'a45f4ff1-735d-4a97-8671-cddb78353bbd',
 'a18547af-ef50-4182-868f-1071902cea8f',
 'e8c514d3-987a-42dd-bc5a-f854d64b5a1a',
 '24a0cd32-b457-4c4a-983c-075a86ae2768',
 'f40ee10c-6c5d-4015-a98b-ce0da560fd19',
 'dabb5900-afe6-4b1f-b633-f5ccccf9fdc0',
 '55784555-f85c-4242-a928-c56357aedf57',
 '4a67958b-8ac5-4c41-ba6b-598860739f57',
 '53c49681-00e0-4ca7-92a2-1a0026c553d5',
 '9a15381f-29ed-493f-890b-2d17578b857f',
 '00f9137f-5d5c-4a53-96f4-5f5d083bc3e4',
 '3d79767f-5fce-4c09-a524-dc64c90e7eba',
 '73da228c-a127-4278-a4ca-ddb8dbcf8604',
 '459a24ca-e265-

In [ ]:
# index_name = "bandhu-db"
# pinecone_index = vector_store.index
# result = pinecone_index.fetch(
#     ids=[ '46ec788b-2193-4dd3-b8c4-3e9f5a56d9a3']
# )
# result

In [17]:
query = "what is the credit system for 4th year"

results = vector_store.similarity_search_with_score(
    query,
    k=3
)

for doc, score in results:
    print(score)
    print(doc.page_content[:1000])
    print("="*50)

0.579717636
 The fourth-year subjects carry only 30 credits. Three
more subjects can be easily accommodated in the fourth
year to earn a Minor.
 To earn a Major in a discipline other than the one in which a student is
registered, the student must secure at least 7.50 CGPA after the third
year, successfully complete at least 20-Credit coursework in that
discipline in the fourth year, in addition to fulfilling all the usual
fourth-year coursework requirements.
 To get two B. Tech (Hons.) or B. Tech. (Research) degrees, a student
fulfil all the coursework requirements of the original discipline in which
the student is registered and spend study another year or more to
complete at least 40 additional Credits of coursework in that discipline
in the fifth year.
5.3.3 The First-Year Curriculum (The Certificate Course)
The first-year curriculum provides the foundation for all the higher-level
courses in the next three and four years of engineering studies. Science
subjects, providing the re

# Retriever

In [18]:
retriever = vector_store.as_retriever(
    search_type = 'mmr',
    search_kwargs = {'k':10,
                     "fetch_k":25,
                     'lambda_mult':0.75}
)

In [19]:
query = "what is the credit system for 4th yea"
docs = retriever.invoke(query)
print(docs)

[Document(metadata={'ModDate': 'D:20260804185939Z', 'Producer': 'iLovePDF', 'file_path': '/content/kiit_rules_updated_merged.pdf', 'page': 41.0, 'source': '/content/kiit_rules_updated_merged.pdf', 'total_pages': 287.0}, page_content='\uf0a7 The fourth-year subjects carry only 30 credits. Three\nmore subjects can be easily accommodated in the fourth\nyear to earn a Minor.\n\uf0b7 To earn a Major in a discipline other than the one in which a student is\nregistered, the student must secure at least 7.50 CGPA after the third\nyear, successfully complete at least 20-Credit coursework in that\ndiscipline in the fourth year, in addition to fulfilling all the usual\nfourth-year coursework requirements.\n\uf0b7 To get two B. Tech (Hons.) or B. Tech. (Research) degrees, a student\nfulfil all the coursework requirements of the original discipline in which\nthe student is registered and spend study another year or more to\ncomplete at least 40 additional Credits of coursework in that discipline\ni

In [20]:
context = "";
for doc in docs:
  context += doc.page_content

print(context)

 The fourth-year subjects carry only 30 credits. Three
more subjects can be easily accommodated in the fourth
year to earn a Minor.
 To earn a Major in a discipline other than the one in which a student is
registered, the student must secure at least 7.50 CGPA after the third
year, successfully complete at least 20-Credit coursework in that
discipline in the fourth year, in addition to fulfilling all the usual
fourth-year coursework requirements.
 To get two B. Tech (Hons.) or B. Tech. (Research) degrees, a student
fulfil all the coursework requirements of the original discipline in which
the student is registered and spend study another year or more to
complete at least 40 additional Credits of coursework in that discipline
in the fifth year.
5.3.3 The First-Year Curriculum (The Certificate Course)
The first-year curriculum provides the foundation for all the higher-level
courses in the next three and four years of engineering studies. Science
subjects, providing the required found

#Langchain Chains -> Augmenatation and generation

In [21]:
from langchain_core.runnables import RunnableSequence

In [22]:
template = PromptTemplate(
    template = """You are an expert Student Handbook Guidelines Assistant.
Your role is to answer student questions using ONLY the information provided in the retrieved handbook context.
Instructions
Carefully read:
The retrieved handbook context
The student's question
Provide a clear, accurate, and student-friendly response based strictly on the handbook rules.
Use simple, direct language. Explain policies in an easy-to-understand way without using complex legal or administrative wording.
Remain fully grounded in the retrieved context:
Do NOT invent policies, penalties, procedures, deadlines, or exceptions.
Do NOT assume information that is not explicitly stated.
Do NOT use outside knowledge.
If faculty names, contact numbers, office details,
or tabular information are present in the retrieved
context, extract them exactly as written.

Do not summarize numbers.
Do not alter phone numbers.
Do not infer missing digits.
If the answer is partially available:
Answer only the portion supported by the context.
Clearly mention what is not specified in the handbook context.
If the retrieved context does not contain enough information:
Say that the handbook does not provide a clear answer.
Suggest checking with the relevant university/college authority or handbook section.
Do NOT fabricate an answer.
Maintain a helpful, professional, and neutral tone.
When appropriate, structure the response as:
Answer concisely (50-150 words).
Only expand if the handbook explicitly provides detailed procedures.
Relevant Rule / Guideline
STRICT RULES:
1. Answer ONLY from the most directly relevant lines.
2. Ignore unrelated handbook text even if retrieved.
3. Do not broaden to nearby policies or committees unless explicitly asked.
4. Every factual statement must be directly supported by retrieved text.
Retrieved Context
{context}
Student Question
{question} """,
    input_variables=['context','question']
)

In [23]:
# template = PromptTemplate(
#     template = """You are an expert Student Handbook Guidelines Assistant.
# Your role is to answer student questions using ONLY the information provided in the retrieved handbook context.
# The context will be provided to you and you have to give an consize to the point answer to it.
# Retrieved Context
# {context}
# Student Question
# {question} """,
#     input_variables=['context','question']
# )

In [24]:
query = "who is the contact person for youth red cross at kiit and how can i contact them ?"
docs = retriever.invoke(query)
context = "";
for doc in docs:
  context += doc.page_content
prompt = template.invoke({'context':context,'question':query})

In [ ]:
import os
os.environ['HUGGINGFACEHUB_ACCESS_TOKEN']=''

In [ ]:
model = ChatGoogleGenerativeAI(model='gemini-3-flash-preview',temperature=1.5,google_api_key ='')

In [ ]:
# from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
# llm = HuggingFaceEndpoint(
#     repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
#     task ='text-generation',
#     huggingfacehub_api_token=os.getenv("")
# )
# model = ChatHuggingFace(llm=llm)

In [29]:
query = "what are the guidelines for PREPARATION OF THE THESIS"
docs = retriever.invoke(query)
context = "";
for doc in docs:
  context += doc.page_content
prompt = template.invoke({'context':context,'question':query})

In [30]:
result = model.invoke(prompt)
print(result.text)

Based on the handbook context, here are the guidelines for preparing your thesis:

### **Answer**
The thesis must be written in precise English (US or UK) and should generally not exceed 200 pages. It must be printed on A4 paper (minimum 85 GSM) using 12-point Times New Roman font with 1.5 line spacing. Each chapter's equations, figures, and tables must be numbered separately, while pages should be numbered consecutively using Arabic numerals. The document must be scanned for plagiarism using "turn-it-in" software. 

The structure must include a supervisor's certificate, six specific chapters (Introduction to Summary), a bibliography, and copies of your peer-reviewed publications. Initially, you should submit five soft-bound copies; after your viva voce, you must submit hard-bound copies with a Rexin cover.

### **Relevant Rule / Guideline**
**1. Technical Specifications (General Guidelines Section 1):**
*   **Paper:** A4 size (297 x 210 mm), weight ≥ 85 GSM.
*   **Font:** Times New Ro

# Implementing the BM25 Keyword Extractor

In [31]:
!pip install -q rank_bm25

In [32]:
from rank_bm25 import BM25Okapi

In [33]:
bm25_docs = [docs.page_content for docs in ndocs]

In [34]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.split()

In [35]:
tokenized_docs = [
    tokenize(doc.page_content)
    for doc in ndocs
]

bm25_index = BM25Okapi(tokenized_docs)

Test

In [36]:
query = "Who is the head of training and placement?"
tokenized_query = tokenize(query)
scores = bm25_index.get_scores(tokenized_query)

In [37]:
top_indices = scores.argsort()[::-1][:5]

In [38]:
results = []

for idx in top_indices:
    results.append({
        "doc": ndocs[idx],
        "score": scores[idx],
        "index": idx
    })

In [39]:
results

[{'doc': Document(metadata={'source': '/content/kiit_rules_updated_merged.pdf', 'file_path': '/content/kiit_rules_updated_merged.pdf', 'page': 247, 'total_pages': 287, 'Producer': 'iLovePDF', 'ModDate': 'D:20260804185939Z', 'text': 'time problems. Open to B.Tech students after fourth and sixth semesters, the summer training is of\none-month duration.\n6.1.2 T&P Guidelines\nFollowing are the guidelines/rules that govern the functioning of the Training & Placement\nDepartment and the roles & responsibilities of the participating students:\n6.1.2.1 Discipline\nDiscipline is given utmost importance by KIIT and therefore only the students who are well-\nbehaved and disciplined throughout their study at KIIT (as per the University disciplinary norms\nset from time to time) are eligible to attend the campus recruitment process. Students who indulge\nin indiscipline on the University campus or during the campus recruitment programs are\ndisqualified. Misbehavior of any kind with any of the Uni

In [40]:
# function for bm25_ query getting

def bm25_search(query, k=5):

    # Tokenize query
    tokenized_query = tokenize(query)

    # Get BM25 scores
    scores = bm25_index.get_scores(tokenized_query)

    # Get top k document indices
    top_indices = scores.argsort()[::-1][:k]

    results = []

    for idx in top_indices:
        results.append({
            "doc": ndocs[idx],
            "score": scores[idx],
            "index": idx
        })

    return results


In [41]:
query = "credit system for 4th year"

results = bm25_search(query, k=5)

for result in results:
    print("Score:", result["score"])
    print(result["doc"].page_content[:500])
    print("=" * 50)

Score: 11.863187360776683
6.4 A separate supplementary examination will be held annually at the
end of each academic year for 1st to 4th year before the start of the next
academic session.
The detail about supplementary examination is given under section no.9
subsequently.
7.0 The Grading System
A student of U.G program shall, at the end of his/her semester program,
receive the grade card for the program mentioning the SGPA according to
∑(CXGP)
Semester grade point average, SGPA =
∑C
Where c= credits of the course, other
Score: 11.782485720222159
those students who not only successfully complete all the course
requirements for the B. Tech. (Hons.) or B. Tech. (Research) degree with
the four open electives in another engineering discipline but also spend an
additional year (the Fifth Year) to complete 40-Credit course requirements
related to that chosen engineering discipline.
A student pursuing the B. Tech. degrees in two engineering disciplines,
thus, needs to choose open electives in

In [42]:
def reciprocal_rank_fusion(
    semantic_docs,
    bm25_results,
    k=60
):

    fused_scores = {}

    # Semantic results
    for rank, doc in enumerate(semantic_docs):

        doc_id = doc.metadata.get("source", "") + str(
            doc.metadata.get("page", "")
        ) + doc.page_content[:100]

        if doc_id not in fused_scores:
            fused_scores[doc_id] = {
                "doc": doc,
                "score": 0
            }

        fused_scores[doc_id]["score"] += 1 / (k + rank + 1)


    # BM25 results
    for rank, result in enumerate(bm25_results):

        doc = result["doc"]

        doc_id = doc.metadata.get("source", "") + str(
            doc.metadata.get("page", "")
        ) + doc.page_content[:100]

        if doc_id not in fused_scores:
            fused_scores[doc_id] = {
                "doc": doc,
                "score": 0
            }

        fused_scores[doc_id]["score"] += 1 / (k + rank + 1)


    # Sort by fused score
    results = sorted(
        fused_scores.values(),
        key=lambda x: x["score"],
        reverse=True
    )

    return results

In [43]:
query = "alcohol consumption laws in kiit"

# Semantic retrieval
semantic_docs = retriever.invoke(query)

# BM25 retrieval
bm25_results = bm25_search(query, k=20)

fused_results = reciprocal_rank_fusion(
    semantic_docs,
    bm25_results
)

# Take top 5
final_docs = [
    result["doc"]
    for result in fused_results[:5]
]

for doc in final_docs:
    print(doc.page_content[:1000])
    print("=" * 50)

4.12.3 Policy Matters on Alcohol & Tobacco
The University campus is declared as “Alcohol /Tobacco free campus”
If a student is found possessing/consuming Tobacco/Alcohol in the University premises or
Hostel, the Disciplinary Committee (DC) will take appropriate action.
4.13 Social Media Guidelines for all Stakeholders including Students
4.13.1 Preliminary
1. KIIT Deemed to be University nurtures a culture of mutual respect for every individual and the
University. Social media communications pervaded every aspect of space and life. The
university recognizes the value of social media and social networking in education. The use of
information technology (disruptive technology) has been optimally leveraged in the
institutional system. Consequently, there arises a mandate for underlining the ethical and legal
challenges.
2. It is considered expedient to spell out the social networking policy/guideline intending to
inform all the members of the university community (student, faculty and staf

In [44]:
def hybrid_search(query, top_k=5):

    semantic_docs = retriever.invoke(query)
    bm25_results = bm25_search(query, k=20)
    fused_results = reciprocal_rank_fusion(
        semantic_docs,
        bm25_results
    )
    return [
        item["doc"]
        for item in fused_results[:top_k]
    ]



In [45]:
query = " where is the office of Dean CSE  in kiit?"

In [46]:
docs = hybrid_search(
    query,
    top_k = 10
)

context = "\n\n".join(
    doc.page_content
    for doc in docs
)

prompt = template.invoke({
    "context": context,
    "question": query
})

print(context)
print("*"*100)

result = model.invoke(prompt)
print(result.text)

University Level Student Counseling Cell
ULSCC Mail ID Contact: chairperson.counselling@kiit.ac.in
Office Address: CRIPP Building, Opposite to the 2nd Gate of Campus-3 (Near the Post Office).
Sl No Name Designation
1 Dr. Pranab Mohapatra, Professor, Psychiatrist, KIMS Chairperson
2 Dr. Kajal Parashar, Asso. Professor, SAS, Dy. Director, GEC, Warden (GH) Member
3 Dr. Ajaya Kumar Parida, Asso. Professor, School of Computer Engg Member
4 Dr. Soumya Mishra, Asst. Professor, Physiology KIMS Member
5 Dr. Arjyadhara Pradhan, Assoc. Professor, SEE Member
6 Dr. Sadhna Sudershana, Assistant Professor, SCA Member
7 Dr. V. Sivasankari, Asso. Professor, KINS Member
8 Dr. Soma Parija, Asst. Professor, Dept of Psychology Member
9 Mr. Siddharth Mishra, Phd Scholar, Clinical Psychology, KIMS Member
10 Dr. Binita Behera , Asst. Professor, School of Law Convener
11 Prof. Damodar Suar, Professor Emeritus Advisor
University Student Grievance Redressal Committee
USGRC Mail ID Contact: grievance.psp@kiit.ac.

# Cross Encoder

In [47]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [48]:
def rerank_documents(query, docs, top_k=5):

    pairs = [
        [query, doc.page_content]
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        doc
        for doc, score in ranked[:top_k]
    ]

In [49]:
def hybrid_search(query, retrieval_k=20, final_k=5):

    semantic_docs = retriever.invoke(query)
    bm25_results = bm25_search(
        query,
        k=retrieval_k
    )
    fused_results = reciprocal_rank_fusion(
        semantic_docs,
        bm25_results
    )
    # Take more documents before reranking
    candidates = [
        item["doc"]
        for item in fused_results[:retrieval_k]
    ]
    # Cross encoder chooses the truly relevant ones
    reranked_docs = rerank_documents(
        query,
        candidates,
        top_k=final_k
    )

    return reranked_docs

In [56]:
query = "what are the email and phone number of Dr. Nitin Varyani?"

docs = hybrid_search(
    query,
    retrieval_k=20,
    final_k=5
)

context = "\n\n".join(
    doc.page_content
    for doc in docs
)

prompt = template.invoke({
    "context": context,
    "question": query
})

result = model.invoke(prompt)

print(result.text)

Based on the retrieved handbook context, here are the contact details for **Dr. Nitin Varyani**:

*   **Email:** nitin.varyanifcs@kiit.ac.in
*   **Phone Number:** The handbook context does not provide a phone number for Dr. Nitin Varyani. It only lists his office room number as B305F.

If you need his phone number, you may wish to check with the relevant university department or the School of Computer Engineering office.

**Relevant Rule / Guideline**
As per the faculty list (Entry 105), the provided details include the faculty name, email address, and office location. Phone numbers are only explicitly listed for specific Points of Contact (POC) in the Exam Cell and Compliance Cell, where Dr. Nitin Varyani is not listed.
